# Neural Architecture Search (NAS) Leve
## Equipe
Rayane Araújo, Júlia Júnior, Marcelo Soares, João Pedro e João Arthur.

> **Objetivo:** Explorar diferentes metaheurísticas (AG, PBIL e Optuna) para automatizar a busca por arquiteturas eficientes de Redes Neurais Convolucionais (CNNs) utilizando o dataset MNIST.

## 0. Bibliotecas e Dataset
Nesta etapa, importamos o `PyTorch` e as bibliotecas essenciais para a manipulação dos dados. Também utilizamos o `torchvision` para baixar o dataset **MNIST**.
Configuramos os `DataLoaders`, que dividirão as imagens em lotes (batches) de 128 imagens para agilizar o treinamento.

In [ ]:
# Verificar e instalar automaticamente dependências ausentes (necessário para o Colab)
import sys
try:
    import optuna
except ImportError:
    import subprocess
    print('Instalando biblioteca Optuna...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import os

# Definir transformações para o MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Se estiver no Colab, baixa no SSD local (/content/data) para evitar gargalo de I/O do Drive. Caso contrário, local
root_dir = '/content/data' if os.path.exists('/content') else './data'
print(f"Diretório de dados configurado em: {root_dir}")

# Carregar conjuntos de treino e teste
train_dataset = torchvision.datasets.MNIST(root=root_dir, train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root=root_dir, train=False, download=True, transform=transform)

# Criar DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f'Tamanho do treino: {len(train_dataset)}, Teste: {len(test_dataset)}')

# Configurar dispositivo de execução (GPU se disponível para o Colab)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo ativo para treinamento: {device}')


## 1. Espaço de Busca e Modelo (CNN Dinâmica)
Criamos a classe `DynamicCNN`. Em um modelo manual, definiríamos cada camada estaticamente. Aqui, a CNN é gerada de acordo com um **cromossomo** (dicionário de configuração).
O vetor de decisão possui:
- `n_layers`: Número de camadas convolucionais (1 a 4).
- `filters`: Lista definindo os canais de cada camada (16, 32, 64 ou 128).
- `activation`: Função de ativação (`ReLU` ou `LeakyReLU`).

*A camada Flatten é calculada passando um dado dummy na rede logo na inicialização, tornando a rede totalmente à prova de erros de dimensão.*

In [ ]:
class DynamicCNN(nn.Module):
    def __init__(self, config, input_shape=(1, 28, 28), num_classes=10):
        super(DynamicCNN, self).__init__()
        self.config = config
        self.features = nn.Sequential()
        
        activation_layer = nn.ReLU if config['activation'] == 'ReLU' else nn.LeakyReLU
        in_channels = input_shape[0]
        
        for i in range(config['n_layers']):
            out_channels = config['filters'][i]
            self.features.add_module(f'conv_{i+1}', nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            self.features.add_module(f'act_{i+1}', activation_layer())
            self.features.add_module(f'pool_{i+1}', nn.MaxPool2d(kernel_size=2, stride=2))
            in_channels = out_channels
            
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_shape)
            flatten_size = self.features(dummy_input).view(-1).shape[0]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flatten_size, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def get_num_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 1.1 Modelo Baseline de Referência

Definimos uma arquitetura de rede neural estática clássica (Baseline) para servir de comparação com as redes descobertas de forma autônoma.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, input_shape=(1, 28, 28), num_classes=10):
        super(BaselineCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_shape)
            flatten_size = self.features(dummy_input).view(-1).shape[0]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flatten_size, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

baseline_model = BaselineCNN()
baseline_params = get_num_parameters(baseline_model)
print(f"Baseline CNN criada com {baseline_params:,} parâmetros.")


## 2. Funções de Treinamento, Avaliação e Fitness
Para as metaheurísticas avaliarem uma rede, precisamos treiná-la. Criamos rotinas rápidas (treinamento de 1 época para exploração do NAS).

**Função Escalarizada (Fitness):**
Para combinar múltiplos objetivos em um só valor avaliativo, definimos a seguinte lógica:
- O valor base é a **acurácia** ($f_1$).
- Se passar de 500.000 parâmetros, aplicamos penalidade grave.
- Se a acurácia for $< 90\%$, subtraímos pontos.
- Modelos mais enxutos (com menos parâmetros) ganham um bônus residual na pontuação.

In [ ]:
def train_model(model, train_loader, epochs=1, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    start_time = time.time()
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()
    return time.time() - start_time

def evaluate_model(model, test_loader, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    correct, total = 0, 0
    inference_times = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_inf = time.time()
            output = model(data)
            inference_times.append((time.time() - start_inf) * 1000)
            
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
    return 100.0 * correct / total, sum(inference_times) / len(inference_times)

def calculate_fitness(accuracy, num_parameters, max_params=500000, min_acc=90.0):
    fitness = accuracy
    if num_parameters > max_params:
        fitness -= (num_parameters / max_params) * 20
    if accuracy < min_acc:
        fitness -= 20
    if num_parameters <= max_params:
        fitness += ((max_params - num_parameters) / max_params) * 2.0 
    return fitness


### Treinamento e Avaliação do Baseline
Treinamos a rede baseline por 1 época e medimos o seu tempo de execução, acurácia de teste e complexidade.

In [ ]:
print("=== TREINANDO E AVALIANDO BASELINE ===")
train_time_bl = train_model(baseline_model, train_loader, epochs=1, device=device)
baseline_acc, baseline_inf = evaluate_model(baseline_model, test_loader, device=device)
print(f"\nResultados do Baseline:")
print(f"- Parâmetros: {baseline_params:,}")
print(f"- Acurácia: {baseline_acc:.2f}%")
print(f"- Tempo de Treino: {train_time_bl:.2f}s")
print(f"- Tempo de Inferência: {baseline_inf:.2f}ms")


## 3. Metaheurística 1: Algoritmo Genético (AG)
A primeira metaheurística é baseada em biologia evolutiva. 
- **População:** Conjunto de arquiteturas (vetores).
- **Crossover:** Pais cruzam para gerar filhos, misturando números de camadas e filtros.
- **Mutação:** Pequena chance de alterar aleatoriamente a arquitetura de um filho.
- **Seleção:** Utilizamos Torneio.
- **Visualização:** A função agora retorna o histórico de melhoria por geração para que possamos plotar depois.

In [ ]:
SEARCH_SPACE = {
    'n_layers': [1, 2, 3, 4],
    'filters': [16, 32, 64, 128],
    'activation': ['ReLU', 'LeakyReLU']
}

# Cache global para evitar re-treinar arquiteturas idênticas entre gerações e sementes
evaluation_cache = {}

def generate_random_individual():
    n_layers = random.choice(SEARCH_SPACE['n_layers'])
    filters = [random.choice(SEARCH_SPACE['filters']) for _ in range(n_layers)]
    activation = random.choice(SEARCH_SPACE['activation'])
    return {'n_layers': n_layers, 'filters': filters, 'activation': activation}

def evaluate_individual(config, train_loader, test_loader, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Criar chave única para o cache baseada nos valores da configuração
    config_key = f"{config['n_layers']}_{config['filters']}_{config['activation']}"
    
    if config_key in evaluation_cache:
        return evaluation_cache[config_key]
        
    model = DynamicCNN(config).to(device)
    num_params = get_num_parameters(model)
    
    if num_params > 500000:
        result = (calculate_fitness(0, num_params), 0.0, num_params)
        evaluation_cache[config_key] = result
        return result
        
    train_time = train_model(model, train_loader, epochs=1, device=device)
    acc, inf_time = evaluate_model(model, test_loader, device)
    fitness = calculate_fitness(acc, num_params)
    
    result = (fitness, acc, num_params)
    evaluation_cache[config_key] = result
    return result

def crossover(parent1, parent2):
    child = {'n_layers': random.choice([parent1['n_layers'], parent2['n_layers']])}
    filters = []
    for i in range(child['n_layers']):
        f1 = parent1['filters'][i] if i < parent1['n_layers'] else random.choice(SEARCH_SPACE['filters'])
        f2 = parent2['filters'][i] if i < parent2['n_layers'] else random.choice(SEARCH_SPACE['filters'])
        filters.append(random.choice([f1, f2]))
    child['filters'] = filters
    child['activation'] = random.choice([parent1['activation'], parent2['activation']])
    return child

def mutate(individual, mutation_rate=0.2):
    if random.random() < mutation_rate:
        individual['n_layers'] = random.choice(SEARCH_SPACE['n_layers'])
        while len(individual['filters']) < individual['n_layers']:
            individual['filters'].append(random.choice(SEARCH_SPACE['filters']))
        individual['filters'] = individual['filters'][:individual['n_layers']]
    for i in range(individual['n_layers']):
        if random.random() < mutation_rate:
            individual['filters'][i] = random.choice(SEARCH_SPACE['filters'])
    if random.random() < mutation_rate:
        individual['activation'] = random.choice(SEARCH_SPACE['activation'])
    return individual

def genetic_algorithm(train_loader, test_loader, pop_size=5, generations=3, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    population = [generate_random_individual() for _ in range(pop_size)]
    best_overall = None
    best_fitness = -float('inf')
    history_fitness = []
    
    for gen in range(generations):
        print(f"\n➤ [AG] Geração {gen+1}/{generations}")
        fitness_scores = []
        for ind in population:
            fit, acc, params = evaluate_individual(ind, train_loader, test_loader, device)
            fitness_scores.append(fit)
            print(f"   [Indivíduo] Fit: {fit:7.2f} | Acc: {acc:5.2f}% | Param: {params}")
            
        best_idx = np.argmax(fitness_scores)
        if fitness_scores[best_idx] > best_fitness:
            best_fitness = fitness_scores[best_idx]
            best_overall = population[best_idx]
            
        history_fitness.append(best_fitness)
        print(f"   👑 Melhor da Geração: Fit {fitness_scores[best_idx]:.2f} -> {population[best_idx]}")
        
        new_population = [population[best_idx]]
        while len(new_population) < pop_size:
            p1 = max(random.sample(list(zip(population, fitness_scores)), 2), key=lambda x: x[1])[0]
            p2 = max(random.sample(list(zip(population, fitness_scores)), 2), key=lambda x: x[1])[0]
            new_population.append(mutate(crossover(p1, p2)))
            
        population = new_population
        
    return best_overall, history_fitness


## 4. Metaheurística 2: Population-Based Incremental Learning (PBIL)
O PBIL é uma transição entre genético e distribuição probabilística. 
Em vez de ter uma população física que cruza, temos um **vetor de probabilidades** (uma "receita" geral).
1. Geramos indivíduos com base na probabilidade atual.
2. Avaliamos os indivíduos.
3. Pegamos o melhor e "puxamos" nosso vetor de probabilidade para ser mais parecido com a arquitetura dele.
Isso promove uma convergência suave.

In [ ]:
class PBIL:
    def __init__(self, search_space, lr=0.1, mut_rate=0.05, mut_shift=0.05):
        self.lr = lr
        self.mut_rate = mut_rate
        self.mut_shift = mut_shift
        
        self.probs = {
            'n_layers': {k: 1.0/len(search_space['n_layers']) for k in search_space['n_layers']},
            'activation': {k: 1.0/len(search_space['activation']) for k in search_space['activation']},
            'filters': [{k: 1.0/len(search_space['filters']) for k in search_space['filters']} for _ in range(max(search_space['n_layers']))]
        }

    def sample_choice(self, prob_dict):
        return np.random.choice(list(prob_dict.keys()), p=list(prob_dict.values()))

    def generate_individual(self):
        n = self.sample_choice(self.probs['n_layers'])
        act = self.sample_choice(self.probs['activation'])
        filts = [self.sample_choice(self.probs['filters'][i]) for i in range(n)]
        return {'n_layers': n, 'filters': filts, 'activation': act}

    def update_probs(self, best):
        for k in self.probs['n_layers']:
            self.probs['n_layers'][k] = self.probs['n_layers'][k]*(1 - self.lr) + (1.0 if k == best['n_layers'] else 0.0)*self.lr
            
        for k in self.probs['activation']:
            self.probs['activation'][k] = self.probs['activation'][k]*(1 - self.lr) + (1.0 if k == best['activation'] else 0.0)*self.lr
            
        for i in range(best['n_layers']):
            for k in self.probs['filters'][i]:
                self.probs['filters'][i][k] = self.probs['filters'][i][k]*(1 - self.lr) + (1.0 if k == best['filters'][i] else 0.0)*self.lr
                
        self._mutate(self.probs['n_layers'])
        self._mutate(self.probs['activation'])
        for i in range(len(self.probs['filters'])): self._mutate(self.probs['filters'][i])
            
    def _mutate(self, prob_dict):
        for k in prob_dict:
            if np.random.random() < self.mut_rate:
                prob_dict[k] += (1 if np.random.random() > 0.5 else -1) * self.mut_shift
                prob_dict[k] = max(0.01, min(0.99, prob_dict[k]))
        total = sum(prob_dict.values())
        for k in prob_dict: prob_dict[k] /= total

def pbil_search(train_loader, test_loader, pop_size=5, generations=3, device=None):
    if device is None: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pbil = PBIL(SEARCH_SPACE)
    best_overall = None
    best_fitness = -float('inf')
    history_fitness = []
    
    for gen in range(generations):
        print(f"\n➤ [PBIL] Geração {gen+1}/{generations}")
        population = [pbil.generate_individual() for _ in range(pop_size)]
        fitness_scores = []
        for ind in population:
            fit, acc, params = evaluate_individual(ind, train_loader, test_loader, device)
            fitness_scores.append(fit)
            print(f"   [Indivíduo] Fit: {fit:7.2f} | Acc: {acc:5.2f}% | Param: {params}")
            
        best_idx = np.argmax(fitness_scores)
        if fitness_scores[best_idx] > best_fitness:
            best_fitness = fitness_scores[best_idx]
            best_overall = population[best_idx]
            
        history_fitness.append(best_fitness)
        print(f"   👑 Melhor da Geração: Fit {fitness_scores[best_idx]:.2f} -> {population[best_idx]}")
        pbil.update_probs(population[best_idx])
        
    return best_overall, history_fitness

# Teste local (Remova o '#' para testar)
# best_pbil, hist_pbil = pbil_search(train_loader, test_loader, pop_size=4, generations=3)

## 5. Metaheurística 3: Optuna (TPE)
Optuna é um framework state-of-art. Ele mapeia os parâmetros em uma árvore e usa probabilidade sequencial para estimar (TPE).
Diferente das heurísticas de população que rodam por gerações, o Optuna trabalha por *Trials* contínuas. A cada nova trial, ele sugere a arquitetura que maximiza nossa função objetivo.

In [ ]:
import optuna

def optuna_objective(trial, train_loader, test_loader, device=None):
    if device is None: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    n_layers = trial.suggest_categorical('n_layers', SEARCH_SPACE['n_layers'])
    activation = trial.suggest_categorical('activation', SEARCH_SPACE['activation'])
    
    filters = [trial.suggest_categorical(f'filters_{i}', SEARCH_SPACE['filters']) for i in range(n_layers)]
    config = {'n_layers': n_layers, 'filters': filters, 'activation': activation}
    
    model = DynamicCNN(config).to(device)
    num_params = get_num_parameters(model)
    
    if num_params > 500000:
        raise optuna.TrialPruned()
        
    train_time = train_model(model, train_loader, epochs=1, device=device)
    acc, inf_time = evaluate_model(model, test_loader, device)
    
    trial.set_user_attr("parameters", num_params)
    trial.set_user_attr("inference_time_ms", inf_time)
    
    return calculate_fitness(acc, num_params)

def optuna_search(train_loader, test_loader, n_trials=10, device=None):
    if device is None: device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    optuna.logging.set_verbosity(optuna.logging.WARNING) # Deixar os logs limpos
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    
    print(f"\n➤ [Optuna] Iniciando {n_trials} Trials...")
    for i in range(n_trials):
        func = lambda trial: optuna_objective(trial, train_loader, test_loader, device)
        study.optimize(func, n_trials=1)
        trial = study.trials[-1]
        if trial.state == optuna.trial.TrialState.COMPLETE:
            print(f"   [Trial {trial.number}] Fit: {trial.value:7.2f} | Config: {trial.params}")
        else:
            print(f"   [Trial {trial.number}] PRUNED (Restrição violada)")
            
    print(f"\n👑 Melhor Optuna: Fit {study.best_value:.2f} -> {study.best_params}")
    
    # Extrair histórico para plot (pegando o melhor valor até a i-ésima trial)
    history_opt = []
    best_so_far = -float('inf')
    for t in study.trials:
        if t.state == optuna.trial.TrialState.COMPLETE and t.value > best_so_far:
            best_so_far = t.value
        history_opt.append(best_so_far if best_so_far != -float('inf') else 0)
        
    return study, history_opt

# Teste local (Remova o '#' para testar)
# study_optuna, hist_optuna = optuna_search(train_loader, test_loader, n_trials=8)

## 6. Resultados Visuais e Comparação (Output Gráfico)
Para finalizar, criamos uma rotina para rodar as três metaheurísticas lado a lado, extrair o histórico de aprendizado delas (`history`), e gerar um gráfico de linhas mostrando como a função de fitness de cada abordagem cresce e se estabiliza ao longo das buscas.

In [ ]:
def plot_convergence(hist_ag, hist_pbil, hist_optuna):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(hist_ag)+1), hist_ag, marker='o', label='Algoritmo Genético (AG)')
    plt.plot(range(1, len(hist_pbil)+1), hist_pbil, marker='s', label='PBIL')
    plt.plot(range(1, len(hist_optuna)+1), hist_optuna, marker='^', linestyle='--', label='Optuna (TPE)')
    plt.title('Comparação de Convergência NAS: Fitness vs Avaliações')
    plt.xlabel('Geração / Trial')
    plt.ylabel('Melhor Fitness Global')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

# ==========================================
# CÉLULA DE EXECUÇÃO FINAL: ANÁLISE DE 5 SEMENTES E EXPORTAÇÃO MD
# (Perfeito para rodar na GPU do Google Colab!)
# ==========================================
POP_SIZE = 12
GENERATIONS = 5
seeds = [1, 2, 3, 4, 5]

results = []
print("=== INICIANDO ANÁLISE COM 5 SEMENTES (AG SO) ===")
for seed in seeds:
    print(f"\n➤ Executando com Semente {seed}...")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    best_config, history_fitness = genetic_algorithm(
        train_loader, test_loader, 
        pop_size=POP_SIZE, 
        generations=GENERATIONS, 
        device=device
    )
    
    # Reavaliar o melhor modelo (TREINAR ANTES DE AVALIAR!)
    model = DynamicCNN(best_config).to(device)
    train_model(model, train_loader, epochs=1, device=device)
    acc, inf_time = evaluate_model(model, test_loader, device=device)
    params = get_num_parameters(model)
    
    results.append({
        'seed': seed,
        'config': best_config,
        'accuracy': acc,
        'params': params,
        'fitness': history_fitness[-1]
    })

# Exibir estatísticas comparativas
print("\n=========================================")
print("RESUMO DAS 5 SEMENTES - PROJETO NAS SO")
print("=========================================")
accuracies = [r['accuracy'] for r in results]
params_counts = [r['params'] for r in results]
fitnesses = [r['fitness'] for r in results]

for r in results:
    print(f"Semente {r['seed']}: Acurácia = {r['accuracy']:.2f}% | Parâmetros = {r['params']:,} | Fitness = {r['fitness']:.2f} | Config = {r['config']}")

print("\n--- Métricas Estatísticas ---")
print(f"Acurácia Média:  {np.mean(accuracies):.2f}% (Desvio Padrão: {np.std(accuracies):.2f}%)")
print(f"Acurácia Máxima: {np.max(accuracies):.2f}% | Acurácia Mínima: {np.min(accuracies):.2f}%")
print(f"Parâmetros Médios: {np.mean(params_counts):,.0f} (Desvio Padrão: {np.std(params_counts):,.0f})")
print(f"Fitness Médio:    {np.mean(fitnesses):.2f} (Desvio Padrão: {np.std(fitnesses):.2f})")
print("\n--- Comparação com o Baseline ---")
print(f"Baseline CNN: Acurácia = {baseline_acc:.2f}% | Parâmetros = {baseline_params:,}")

# ==========================================
# SALVAR RESULTADOS EM ARQUIVO MARKDOWN (.md)
# ==========================================
md_filename = "resultados_seeds.md"
with open(md_filename, "w", encoding="utf-8") as f:
    f.write("# Relatório de Experimentos - Neural Architecture Search Leve (SO)\n\n")
    f.write(f"- **Data/Hora da Execução**: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"- **Dispositivo (Hardware)**: {device}\n\n")
    
    f.write("## 1. Baseline de Referência (CNN Estática)\n")
    f.write("Uma CNN simples com 2 camadas convolucionais (32 e 64 filtros) e pooling.\n\n")
    f.write(f"- **Acurácia**: {baseline_acc:.2f}%\n")
    f.write(f"- **Parâmetros**: {baseline_params:,}\n")
    f.write(f"- **Tempo de Treino**: {train_time_bl:.2f}s\n")
    f.write(f"- **Tempo de Inferência**: {baseline_inf:.2f}ms\n\n")
    
    f.write("## 2. Resultados das 5 Sementes (Algoritmo Genético SO)\n")
    f.write(f"Busca executada com `POP_SIZE = {POP_SIZE}` e `GENERATIONS = {GENERATIONS}`.\n\n")
    f.write("| Semente | Acurácia (%) | Parâmetros | Fitness Final | Configuração da CNN (Filtros, Camadas, Ativação) |\n")
    f.write("| :---: | :---: | :---: | :---: | :--- |\n")
    for r in results:
        f.write(f"| {r['seed']} | {r['accuracy']:.2f}% | {r['params']:,} | {r['fitness']:.2f} | `{r['config']}` |\n")
    f.write("\n")
    
    f.write("## 3. Análise Estatística Combinada\n\n")
    f.write(f"- **Acurácia Média**: {np.mean(accuracies):.2f}% (Desvio Padrão: {np.std(accuracies):.2f}%)\n")
    f.write(f"- **Acurácia Máxima**: {np.max(accuracies):.2f}% | **Acurácia Mínima**: {np.min(accuracies):.2f}%\n")
    f.write(f"- **Média de Parâmetros**: {np.mean(params_counts):,.0f} (Desvio Padrão: {np.std(params_counts):,.0f})\n")
    f.write(f"- **Fitness Médio**: {np.mean(fitnesses):.2f} (Desvio Padrão: {np.std(fitnesses):.2f})\n\n")
    
    f.write("## 4. Discussão Inicial\n")
    f.write("Este arquivo foi gerado automaticamente para facilitar a análise de convergência e trade-off entre complexidade (parâmetros) e acurácia da rede. Compartilhe este arquivo com o seu assistente de IA para gerar insights comparativos detalhados.\n")

print(f"\n[Sucesso] Resultados exportados para o arquivo: '{md_filename}'")
